# 05 Models — CatBoost — Women's

CatBoost as an additional gradient boosting variant. CatBoost handles missing values natively and uses ordered boosting to reduce overfitting.

**Inputs** (from S3 `04_preprocessing/womens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/catboost/womens/`):
- `oof_predictions.parquet`, `stage1_predictions.parquet`, `stage2_predictions.parquet`, `cv_results.parquet`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

try:
    from catboost import CatBoostRegressor, Pool
except:
    !pip install catboost
    from catboost import CatBoostRegressor, Pool

try:
    import optuna
except:
    !pip install optuna
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

import catboost
print(f"CatBoost version: {catboost.__version__}")
print(f"Optuna version: {optuna.__version__}")

CatBoost version: 1.2.10
Optuna version: 4.7.0


#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "womens"
STAGE = "05_models/catboost"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

Training data: (3434, 173)
Stage 1 grid: (258131, 171)
Stage 2 grid: (65703, 170)
Features: 58


## 2. Optuna Hyperparameter Tuning

Use 4-fold Stage 1 cross-validation (2022-2025) with reduced trials to find optimal CatBoost hyperparameters.

In [6]:
train_original = train[train['TeamA'] < train['TeamB']].copy()
stage1_folds = [2022, 2023, 2024, 2025]

def objective(trial):
    params = {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'depth': trial.suggest_int('depth', 2, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'bootstrap_type': 'Bernoulli',
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'iterations': 500,
        'early_stopping_rounds': 50,
        'random_seed': 42,
        'verbose': 0,
    }
    
    fold_scores = []
    for fold in stage1_folds:
        train_mask = train['Fold'] != fold
        val_mask = train_original['Fold'] == fold
        
        X_tr = train.loc[train_mask, feature_cols]
        y_tr = train.loc[train_mask, 'Label']
        X_va = train_original.loc[val_mask, feature_cols]
        y_va = train_original.loc[val_mask, 'Label']
        
        if len(X_va) == 0:
            continue
        
        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
        
        preds = model.predict(X_va).clip(0, 1)
        fold_scores.append(brier_score_loss(y_va, preds))
    
    return np.mean(fold_scores)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best Stage 1 Brier: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]

Best Stage 1 Brier: 0.1341
Best params: {'depth': 7, 'learning_rate': 0.17151565373657812, 'l2_leaf_reg': 0.35140641859177174, 'subsample': 0.5452960382391238, 'random_strength': 7.982725499896262}


## 3. Set Tuned CatBoost Hyperparameters

In [7]:
best = study.best_params

# best = {
#     'loss_function': 'RMSE',
#     'eval_metric': 'RMSE',
#     'depth': 8,
#     'learning_rate': 0.16627390575689183,
#     'l2_leaf_reg': 0.630153856471159,
#     'subsample': 0.6803395985027783,
#     'bootstrap_type': 'Bernoulli',
#     'random_strength': 2.0217257874274597,
#     'iterations': 500,
#     'early_stopping_rounds': 50,
#     'random_seed': 42,
#     'verbose': 0,
# }

In [8]:
cb_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'depth': best['depth'],
    'learning_rate': best['learning_rate'],
    'l2_leaf_reg': best['l2_leaf_reg'],
    'subsample': best['subsample'],
    'bootstrap_type': 'Bernoulli',
    'random_strength': best['random_strength'],
    'iterations': 500,
    'early_stopping_rounds': 50,
    'random_seed': 42,
    'verbose': 0,
}

print("Tuned CatBoost parameters:")
for k, v in cb_params.items():
    print(f"  {k}: {v}")

Tuned CatBoost parameters:
  loss_function: RMSE
  eval_metric: RMSE
  depth: 7
  learning_rate: 0.17151565373657812
  l2_leaf_reg: 0.35140641859177174
  subsample: 0.5452960382391238
  bootstrap_type: Bernoulli
  random_strength: 7.982725499896262
  iterations: 500
  early_stopping_rounds: 50
  random_seed: 42
  verbose: 0


## 4. LOGO Cross-Validation (with Tuned Params)

In [9]:
folds = sorted(train['Fold'].unique())

oof_preds = []
cv_results = []

for fold in folds:
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols]
    y_train = train.loc[train_mask, 'Label']
    X_val = train_original.loc[val_mask, feature_cols]
    y_val = train_original.loc[val_mask, 'Label']
    
    if len(X_val) == 0:
        continue
    
    model = CatBoostRegressor(**cb_params)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        verbose=False
    )
    
    # CatBoostRegressor returns raw values — clip to [0, 1] for valid probabilities
    preds = model.predict(X_val).clip(0, 1)
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': model.best_iteration_
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}, BestRound={model.best_iteration_}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

  Fold 1998: Brier=0.1620, Games=63, BestRound=93
  Fold 1999: Brier=0.1323, Games=63, BestRound=51
  Fold 2000: Brier=0.1441, Games=63, BestRound=51
  Fold 2001: Brier=0.1552, Games=63, BestRound=71
  Fold 2002: Brier=0.1317, Games=63, BestRound=63
  Fold 2003: Brier=0.1160, Games=63, BestRound=76
  Fold 2004: Brier=0.1670, Games=63, BestRound=70
  Fold 2005: Brier=0.1569, Games=63, BestRound=19
  Fold 2006: Brier=0.1324, Games=63, BestRound=18
  Fold 2007: Brier=0.1781, Games=63, BestRound=42
  Fold 2008: Brier=0.0934, Games=63, BestRound=231
  Fold 2009: Brier=0.1454, Games=63, BestRound=95
  Fold 2010: Brier=0.1483, Games=63, BestRound=15
  Fold 2011: Brier=0.1312, Games=63, BestRound=71
  Fold 2012: Brier=0.1228, Games=63, BestRound=69
  Fold 2013: Brier=0.1316, Games=63, BestRound=69
  Fold 2014: Brier=0.1210, Games=63, BestRound=47
  Fold 2015: Brier=0.1360, Games=63, BestRound=25
  Fold 2016: Brier=0.1637, Games=63, BestRound=47
  Fold 2017: Brier=0.1395, Games=63, BestRound=41

In [ ]:
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
weighted_brier = np.average(cv_df.sort_values('Fold')['BrierScore'], weights=np.arange(1, len(cv_df) + 1))
print(f"Weighted Mean Brier: {weighted_brier:.4f}")

## 5. Train Final Model and Calibrate

In [11]:
final_rounds = int(cv_df['BestRound'].median())
print(f"Final model rounds: {final_rounds}")

final_params = cb_params.copy()
final_params['iterations'] = final_rounds
final_params.pop('early_stopping_rounds', None)

final_model = CatBoostRegressor(**final_params)
final_model.fit(train[feature_cols], train['Label'], verbose=False)

# Calibration
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])
print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")

Final model rounds: 47
OOF Brier (raw): 0.1403
OOF Brier (calibrated): 0.1368


## 6. Generate Predictions

In [12]:
# CatBoostRegressor returns raw values — clip to [0, 1]
stage1['Pred_catboost'] = final_model.predict(stage1[feature_cols]).clip(0, 1)
stage1['Pred_catboost_calibrated'] = calibrator.predict(stage1['Pred_catboost'].values).clip(0.02, 0.98)

stage2['Pred_catboost'] = final_model.predict(stage2[feature_cols]).clip(0, 1)
stage2['Pred_catboost_calibrated'] = calibrator.predict(stage2['Pred_catboost'].values).clip(0.02, 0.98)

print(f"Stage 1 predictions range: [{stage1['Pred_catboost_calibrated'].min():.3f}, {stage1['Pred_catboost_calibrated'].max():.3f}]")
print(f"Stage 2 predictions range: [{stage2['Pred_catboost_calibrated'].min():.3f}, {stage2['Pred_catboost_calibrated'].max():.3f}]")

stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_catboost_calibrated'])
    print(f"Stage 1 Brier (calibrated): {s1_brier:.4f}")

Stage 1 predictions range: [0.020, 0.980]
Stage 2 predictions range: [0.020, 0.980]
Stage 1 Brier (calibrated): 0.0819


## 7. Save Outputs

In [13]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_catboost', 'Pred_catboost_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_catboost', 'Pred_catboost_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/catboost/womens/oof_predictions.parquet ((1717, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/catboost/womens/stage1_predictions.parquet ((258131, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/catboost/womens/stage2_predictions.parquet ((65703, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/catboost/womens/cv_results.parquet ((27, 4))


## 8. Summary

In [ ]:
print("=" * 60)
print("CATBOOST MODEL SUMMARY — WOMEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
weighted_brier = np.average(cv_df.sort_values('Fold')['BrierScore'], weights=np.arange(1, len(cv_df) + 1))
print(f"Weighted Mean Brier: {weighted_brier:.4f}")
print(f"Final model rounds: {final_rounds}")